# Notebook 27 - Gate G6: recovery efficiency

Pre-registered. Frozen 40% structures from the 17b/20b registries; no new selection, no test access. Measures recovery units to a pooled-plateau threshold across 5 methods x 3 seeds x 2 architectures, with censoring. Resumable per (architecture, method, seed).

In [ ]:
# NB27 (Gate G6): recovery-efficiency factorial on frozen 40% structures.
# No new selection. No test access. 5 methods x 3 seeds x 2 architectures.
from google.colab import drive; drive.mount("/content/drive", force_remount=False)
import os, sys, json, math
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, yaml
import matplotlib.pyplot as plt
from scipy.stats import kruskal

REPO = Path("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
os.chdir(REPO); sys.path.insert(0, str(REPO))
from src.saber.bridge_ciciot import load_bridge
from src.saber.taxonomy import ciciot2023_taxonomy, DEFAULT_COST_PROFILES
from src.saber.surgery import prune_cnn1d_channels
from src.saber.metrics import full_model_audit, action_weighted_boundary_inversion_rate

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
R = REPO / "results/saber"
OUT = R / "27_recovery_efficiency"; OUT.mkdir(parents=True, exist_ok=True)
SABER_CFG = yaml.safe_load(open(REPO / "config/saber.yaml"))

METHODS = ["random", "magnitude", "taylor", "fisher", "saber_v2"]
SEEDS = [11, 23, 37]
E_MAX = {"shallow": 8, "deep": 6}
MIN_W = {"shallow": int(SABER_CFG["groups"]["minimum_remaining_per_layer"]), "deep": 8}
SUBSET_FRACTION = 0.10
FAM_PLATEAU_FRACTION = 0.95
B2A_GUARD = 0.05
RUN_ARCHITECTURES = ["shallow", "deep"]   # set to one arch for staged runs

PREREG = {
    "gate": "G6_recovery_efficiency",
    "question": "Does channel selection change the COST of recovery, given it does not change the plateau?",
    "primary_outcome": "recovery units (passes over the frozen 10% subset) to reach tau",
    "tau": ("family macro-F1 >= %.2f x pooled plateau (median across methods of per-method "
            "median final value), with benign_to_attack <= %.2f at the crossing pass" %
           (FAM_PLATEAU_FRACTION, B2A_GUARD)),
    "censoring": "runs not reaching tau by E_MAX recorded as censored at E_MAX+1",
    "seeds": SEEDS, "methods": METHODS, "e_max": E_MAX,
    "G6a_pass": "Kruskal-Wallis p<0.05 across methods AND median spread >= 1 unit, in >=1 architecture",
    "G6b_pass": "saber_v2 strictly lowest median units in >=1 architecture and within 1 unit elsewhere",
    "secondary": ["mean family macro-F1 across passes (recovery AUC)", "mean AWBIR across passes"],
    "no_test_access": True, "no_new_selection": True,
}
(OUT / "G6_PREREGISTRATION.json").write_text(json.dumps(PREREG, indent=2))
print(json.dumps(PREREG, indent=2))

TRAIN_LOADER, VAL_LOADER, _TEST_UNUSED, SHALLOW_TEACHER, CLASS_NAMES = load_bridge()
taxonomy = ciciot2023_taxonomy(CLASS_NAMES)
robust_graph = pd.read_csv(R / "14_risk_graph/asvg_edges_robust.csv")
N_CLASSES = len(CLASS_NAMES)

class DeepCNN1D(nn.Module):
    def __init__(self, n_classes=34):
        super().__init__()
        def blk(i, o): return [nn.Conv1d(i, o, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(o)]
        self.conv = nn.Sequential(*blk(1, 64), *blk(64, 128), nn.MaxPool1d(2),
                                  *blk(128, 128), *blk(128, 256))
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(256, n_classes)
    def forward(self, x):
        if x.dim() == 2: x = x.unsqueeze(1)
        return self.head(self.pool(self.conv(x.float())).squeeze(-1))

TEACHERS = {"shallow": SHALLOW_TEACHER.to(DEVICE).eval()}
if "deep" in RUN_ARCHITECTURES:
    dt = DeepCNN1D(N_CLASSES)
    dt.load_state_dict(torch.load(REPO / "models/ciciot2023/deepcnn1d_g5_seed0.pt",
                                  map_location="cpu", weights_only=False)["state_dict"])
    TEACHERS["deep"] = dt.to(DEVICE).eval()

# ---- frozen evaluation subsample (stratified, seed 0, identical for every run) ----
Xv, Yv = VAL_LOADER.dataset.tensors
VAL_Y_ALL = Yv.numpy()
rng = np.random.default_rng(0)
idx = np.concatenate([rng.permutation(np.where(VAL_Y_ALL == c)[0])[:4000]
                      for c in range(N_CLASSES) if (VAL_Y_ALL == c).sum() > 0])
EX_X = Xv[idx].to(DEVICE); EX_Y = VAL_Y_ALL[idx]
print("evaluation subsample rows:", len(idx))
EXAMPLE_INPUT = Xv[:8].float().to(DEVICE)

T_LOGITS = {}
for a, m in TEACHERS.items():
    with torch.no_grad():
        T_LOGITS[a] = torch.cat([m(EX_X[i:i+8192]).cpu() for i in range(0, len(EX_X), 8192)]).numpy()

def audit_of(model, arch):
    model.eval()
    with torch.no_grad():
        lg = torch.cat([model(EX_X[i:i+8192]).cpu() for i in range(0, len(EX_X), 8192)]).numpy()
    a = full_model_audit(lg, EX_Y, taxonomy, DEFAULT_COST_PROFILES)
    aw, _ = action_weighted_boundary_inversion_rate(T_LOGITS[arch], lg, EX_Y, robust_graph)
    a["awbir"] = float(aw)
    return a

train_y = TRAIN_LOADER.dataset.tensors[1].numpy()
counts = np.bincount(train_y, minlength=N_CLASSES)
w = np.zeros_like(counts, dtype=np.float64)
w[counts > 0] = 1.0 / np.sqrt(counts[counts > 0]); w[counts > 0] /= w[counts > 0].mean()
CLASS_W = torch.tensor(w, dtype=torch.float32, device=DEVICE)
N_TRAIN = len(TRAIN_LOADER.dataset)

def removed_path(arch, method):
    if arch == "shallow":
        return R / f"17b_calibrated_checkpoint_freeze/{method}_r40cal_removed_groups.csv"
    return R / f"20b_depth_checkpoint_freeze/{method}_minimal_r40_removed_groups.csv"

def raw_student(arch, method):
    rm = pd.read_csv(removed_path(arch, method))
    pm = {str(l): sorted(g["channel_index"].astype(int).tolist())
          for l, g in rm.groupby("module_path")}
    st, _ = prune_cnn1d_channels(TEACHERS[arch], pm, EXAMPLE_INPUT,
                                 minimum_remaining_per_layer=MIN_W[arch])
    return st.to(DEVICE)

CURVE = OUT / "recovery_curves.csv"
rows = pd.read_csv(CURVE).to_dict("records") if CURVE.exists() else []
done = {(r["architecture"], r["method"], r["seed"]) for r in rows}

for arch in RUN_ARCHITECTURES:
    for method in METHODS:
        for seed in SEEDS:
            if (arch, method, seed) in done: continue
            torch.manual_seed(seed); np.random.seed(seed)
            student = raw_student(arch, method)
            a0 = audit_of(student, arch)
            rows.append({"architecture": arch, "method": method, "seed": seed, "unit": 0,
                         **{k: float(a0[k]) for k in ["fine_macro_f1", "family_macro_f1",
                            "awbir", "attack_to_benign_rate", "benign_to_attack_rate",
                            "hsr_balanced_soc", "ece15"]}})
            g = torch.Generator().manual_seed(seed)
            sub = torch.randperm(N_TRAIN, generator=g)[: int(N_TRAIN * SUBSET_FRACTION)]
            loader = torch.utils.data.DataLoader(
                torch.utils.data.Subset(TRAIN_LOADER.dataset, sub.tolist()),
                batch_size=1024, shuffle=True,
                generator=torch.Generator().manual_seed(seed))
            opt = torch.optim.Adam(student.parameters(), lr=1e-3)
            lossf = nn.CrossEntropyLoss(weight=CLASS_W)
            for unit in range(1, E_MAX[arch] + 1):
                student.train()
                for xb, yb in loader:
                    opt.zero_grad()
                    lossf(student(xb.float().to(DEVICE)), yb.to(DEVICE)).backward()
                    opt.step()
                a = audit_of(student, arch)
                rows.append({"architecture": arch, "method": method, "seed": seed, "unit": unit,
                             **{k: float(a[k]) for k in ["fine_macro_f1", "family_macro_f1",
                                "awbir", "attack_to_benign_rate", "benign_to_attack_rate",
                                "hsr_balanced_soc", "ece15"]}})
                pd.DataFrame(rows).to_csv(CURVE, index=False)
            print(f"{arch} {method} seed{seed}: famF1 {a0['family_macro_f1']:.3f} -> "
                  f"{a['family_macro_f1']:.3f}, awbir {a['awbir']:.3f}")

df = pd.DataFrame(rows)

# ---- tau, units-to-tau, gate ----
summary, gate_arch = [], {}
for arch in sorted(df["architecture"].unique()):
    d = df[df["architecture"] == arch]
    finals = d[d["unit"] == E_MAX[arch]].groupby("method")["family_macro_f1"].median()
    tau = FAM_PLATEAU_FRACTION * float(finals.median())
    for method in METHODS:
        for seed in SEEDS:
            run = d[(d["method"] == method) & (d["seed"] == seed)].sort_values("unit")
            if run.empty: continue
            hit = run[(run["unit"] > 0) & (run["family_macro_f1"] >= tau) &
                      (run["benign_to_attack_rate"] <= B2A_GUARD)]
            units = int(hit["unit"].iloc[0]) if len(hit) else E_MAX[arch] + 1
            summary.append({"architecture": arch, "method": method, "seed": seed, "tau": tau,
                            "units_to_tau": units, "censored": bool(len(hit) == 0),
                            "auc_family_f1": float(run[run["unit"] > 0]["family_macro_f1"].mean()),
                            "mean_awbir": float(run[run["unit"] > 0]["awbir"].mean())})
    s = pd.DataFrame(summary); s = s[s["architecture"] == arch]
    groups = [s[s["method"] == m]["units_to_tau"].values for m in METHODS]
    try: p = float(kruskal(*groups).pvalue)
    except Exception: p = float("nan")
    med = s.groupby("method")["units_to_tau"].median()
    gate_arch[arch] = {"tau": tau, "kruskal_p": p,
                       "median_units": {m: float(med.get(m, float("nan"))) for m in METHODS},
                       "median_spread": float(med.max() - med.min()),
                       "best_method": str(med.idxmin()),
                       "G6a": bool(p == p and p < 0.05 and (med.max() - med.min()) >= 1.0)}
sm = pd.DataFrame(summary); sm.to_csv(OUT / "units_to_tau.csv", index=False)

g6a = any(v["G6a"] for v in gate_arch.values())
sab_best = [a for a, v in gate_arch.items() if v["best_method"] == "saber_v2"
            and list(v["median_units"].values()).count(v["median_units"]["saber_v2"]) == 1]
others_ok = all(gate_arch[a]["median_units"]["saber_v2"] -
                min(gate_arch[a]["median_units"].values()) <= 1.0 for a in gate_arch)
gate = {"gate": "G6_recovery_efficiency", "G6a_passed": bool(g6a),
        "G6b_passed": bool(len(sab_best) > 0 and others_ok),
        "per_architecture": gate_arch, "prereg": PREREG}
(OUT / "G6_recovery_efficiency_gate.json").write_text(json.dumps(gate, indent=2))
print(json.dumps(gate, indent=2))

fig, axes = plt.subplots(1, len(gate_arch), figsize=(5.2 * len(gate_arch), 3.2), squeeze=False)
for ax, arch in zip(axes[0], sorted(gate_arch)):
    d = df[df["architecture"] == arch]
    for m in METHODS:
        mm = d[d["method"] == m].groupby("unit")["family_macro_f1"].median()
        ax.plot(mm.index, mm.values, marker="o", lw=1.3, label=m)
    ax.axhline(gate_arch[arch]["tau"], color="0.4", ls="--", lw=0.9)
    ax.set_title(f"{arch}: recovery curve (median of {len(SEEDS)} seeds)")
    ax.set_xlabel("recovery units"); ax.set_ylabel("family macro-F1"); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "G6_recovery_curves.png", dpi=200); plt.show()
print("done ->", OUT)
